# 🔥 FireWatch — FULL training on a Colab GPU

Trains the **TensorFlow / KerasHub RetinaNet** on the **full D-Fire dataset** (17,221 train
images, validated on the 4,306 test images) using Colab's free GPU — ~minutes/epoch instead
of ~3.5 hours/epoch on a laptop CPU.

### Before you start (one-time)
1. **Runtime → Change runtime type → GPU → Save.**
2. Get the **full D-Fire** into your Google Drive. Easiest options:
   - **(A)** Drag the whole `D-Fire` folder into **My Drive** (so you have `MyDrive/D-Fire/train` and `MyDrive/D-Fire/test`).
   - **(B)** Or zip it to `dfire_full.zip`, upload that to My Drive, and the data cell unzips it.
3. Make sure your latest code is pushed to GitHub `master` (this notebook clones it).

Then **Runtime → Run all**. The best model auto-saves to your Drive as you train, so a Colab
disconnect never loses progress.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'No GPU! Runtime > Change runtime type > GPU, then rerun.'

## 2. Get the code + install KerasHub
Clones the `master` branch (all the code lives there). If pip shows a **RESTART RUNTIME**
button, click it, then re-run this cell.

In [ ]:
import os
if not os.path.isdir('fire-detection'):
    !git clone --depth 1 -b master https://github.com/01End/fire-detection.git
%cd fire-detection
!pip install -q -U keras-hub opencv-python-headless
import keras_hub, keras
print('keras', keras.__version__, '| keras_hub', keras_hub.__version__)

## 3. Mount Drive and locate the full dataset
Authorize Drive when prompted. This finds `D-Fire` either as a folder or a zip in My Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA = '/content/drive/MyDrive/D-Fire'        # option (A): folder in My Drive
ZIP  = '/content/drive/MyDrive/dfire_full.zip'  # option (B): a zip in My Drive
if not os.path.isdir(os.path.join(DATA, 'train', 'images')) and os.path.exists(ZIP):
    !mkdir -p /content/data && unzip -q -o "$ZIP" -d /content/data
    # adjust if the zip's top folder isn't named D-Fire:
    DATA = '/content/data/D-Fire'
assert os.path.isdir(os.path.join(DATA, 'train', 'images')), f'no train/images under {DATA}'
print('DATA =', DATA)
print('train images:', len(os.listdir(DATA + '/train/images')),
      '| test images:', len(os.listdir(DATA + '/test/images')))

## 4. Train on the FULL dataset 🚀
Saves the best model **straight to your Drive** (`fire_retinanet_full.weights.h5`) every time
validation improves — so if Colab disconnects, your best model is already safe. ~10–15 min per
epoch on a T4. Lower `--epochs` if you're tight on Colab time; the best checkpoint is kept either way.

In [ ]:
OUT = '/content/drive/MyDrive/fire_retinanet_full.weights.h5'
!python -m training.tf_train \
    --data "{DATA}" \
    --train-split train \
    --val-split test \
    --epochs 10 \
    --batch-size 8 \
    --image-size 512 \
    --lr 0.0005 \
    --out "{OUT}"

## 5. Measure accuracy on the held-out test set 📊
Precision / recall / F1 @ IoU 0.5 on test images the model never trained on (fast on GPU).

In [ ]:
!python -m training.tf_eval \
    --model "{OUT}" \
    --data "{DATA}/test" \
    --image-size 512 --score-thr 0.4 --limit 500

## 6. See it work — annotated detections 🔥

In [ ]:
import glob, shutil
os.makedirs('val_sample', exist_ok=True)
# pick a few TEST images that actually contain fire/smoke (non-empty labels)
picked = 0
for lf in sorted(glob.glob(DATA + '/test/labels/*.txt')):
    if os.path.getsize(lf) > 0:
        stem = os.path.splitext(os.path.basename(lf))[0]
        ip = DATA + '/test/images/' + stem + '.jpg'
        if os.path.exists(ip):
            shutil.copy(ip, 'val_sample/'); picked += 1
    if picked >= 8:
        break

!python -m firewatch.cli detect \
    --model "{OUT}" \
    --input val_sample \
    --backend tf --arch retinanet \
    --image-size 512 --score-thr 0.4 \
    --out detect_out

from IPython.display import Image, display
for p in sorted(glob.glob('detect_out/*'))[:8]:
    print(p); display(Image(p))